In [9]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
from scipy.stats import maxwell
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [2]:
N = 5000
sigma = 1.0
#define independed velocity components
vx = np.random.normal(0, sigma, N)
vy = np.random.normal(0, sigma, N)
vz = np.random.normal(0, sigma, N)

v = np.sqrt(vx**2 + vy**2 + vz**2)

In [3]:
#plot gaussian distribution of velocity components
scatter = go.Scatter3d(
    x=vx,
    y=vy,
    z=vz,
    mode="markers",
    marker=dict(
        size=2,
        color="royalblue",
        opacity=0.25
    ), 
    name = "Particles"
)

In [4]:
#spherical shells 
theta = np.linspace(0, np.pi, 50)
phi = np.linspace(0, 2 * np.pi, 50)
X = np.outer(np.sin(theta), np.cos(phi))
Y = np.outer(np.sin(theta), np.sin(phi))
Z = np.outer(np.cos(theta), np.ones_like(phi))



In [5]:
r = 1.5 

Xs = r * X
Ys = r * Y
Zs = r * Z

dr = 0.05
inside_shell = np.abs(v - r) < dr

count = np.sum(inside_shell)

colors = np.where(
    inside_shell,
    "yellow",
    "royalblue"
)

marker=dict(
    size=2,
    color=colors
)
surface = go.Surface(
    x=Xs,
    y=Ys,
    z=Zs,
    opacity=0.2,
    showscale=False,
    name="Shell"
)


In [6]:
fig = go.Figure(data=[scatter, surface])
fig.show()

In [7]:
fig.write_html("maxwell_particles.html")

### HOW CHANGE IN DIMENSIONS SHAPE THE DISTRIBUTION?

In [8]:
#1D -> Normal
n=20000
x = np.random.normal(0, sigma, n)
fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=x,
        histnorm="probability density",
        nbinsx=50,
        name="Simulation"
    )
)

xx = np.linspace(-4,4,300)

pdf = (1/(sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * (xx / sigma)**2)

fig.add_trace(
    go.Scatter(
        x=xx,
        y=pdf,
        mode="lines",
        name="Normal PDF"
    )
)

fig.update_layout(
    title="1D: Normal Distribution",
    xaxis_title="x",
    yaxis_title="Density"
)

fig.show()

### Restricted Boltzmann Machine (RBM)

In [11]:
np.random.seed(0)

n_visible = 4
n_hidden = 2

# weights
W = np.random.randn(n_visible, n_hidden)

# biases
a = np.random.randn(n_visible)
b = np.random.randn(n_hidden)

def energy(v, h):
    return -np.dot(v, a) - np.dot(h, b) - np.dot(v, np.dot(W, h))

#Boltzmann activation: P(hi = 1 | v) = sigmoid(bi + sum_j Wij * vj)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sample_hidden(v):
    p = sigmoid(b + W.T @ v)
    return (np.random.rand(n_hidden) < p).astype(int)

def sample_visible(h):
    p = sigmoid(a + W @ h)
    return (np.random.rand(n_visible) < p).astype(int)

v = np.random.randint(0,2,n_visible)
h = np.random.randint(0,2,n_hidden)